# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasmeenmh90-beep/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Classification, clustering, ranking, or scoring — which one, and why? Binary classification feeding a ranked scoring queue. The core prediction is binary (declining or not), but the actual deliverable is a ranked list, since a reviewer can only act on a limited number of pages at a time.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


What would you predict? Where does that label come from? The target is a proxy: is_declining_label = (trend_direction == "down"), computed from the current window's trend bucket — not yet a true future outcome. A stronger future-looking version would be: prior 90 days of features → decline over next 30 days. Using the current proxy now for framing, flagged honestly as not final.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Precision@50 (and Precision@20 as a secondary check). This matches how the output gets used — a reviewer checks a fixed number of top-ranked pages, so what matters is how many of the top 50 flagged are actually right, not generic accuracy across all 30,000 rows.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

REPO_URL = "https://github.com/yasmeenmh90-beep/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship-starter


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"{len(df)} rows, one row per content_id (one page)")
display(df[["content_id", "client_id", "impressions_90d", "trend_direction", "avg_position", "ctr"]].head(5))

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True).round(3))

30000 rows, one row per content_id (one page)


,content_id,client_id,impressions_90d,trend_direction,avg_position,ctr
0,content_304f48230142,client_f369cb89fc,3803,down,10.6,0.76
1,content_a1fb4e703a9e,client_4e07408562,15320,down,20.3,0.05
2,content_9aa793d4d895,client_7f2253d7e2,12581,down,36.5,0.09
3,content_331d6c4de07b,client_19581e27de,11751,stable,6.2,0.49
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down,44.0,0.13


is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


In [3]:

# ML-03 — Validate the prediction unit and label

print("=== DATASET VALIDATION ===")
print("Total records:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())
print("Unique clients:", df["client_id"].nunique())

print("\n=== LABEL DISTRIBUTION ===")
print(df["is_declining_label"].value_counts())
print(df["is_declining_label"].value_counts(normalize=True).round(4))

print("\n=== VALIDATION CHECKS ===")
assert len(df) == 30000, "Unexpected dataset size"
assert df["content_id"].is_unique, "Duplicate content IDs detected"
assert df["is_declining_label"].isin([0, 1]).all(), "Invalid labels"

print("PASS: Dataset grain and binary labels validated.")


=== DATASET VALIDATION ===
Total records: 30000
Unique content IDs: 30000
Duplicate content IDs: 0
Unique clients: 32

=== LABEL DISTRIBUTION ===
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
is_declining_label
1    0.5421
0    0.4579
Name: proportion, dtype: float64

=== VALIDATION CHECKS ===
PASS: Dataset grain and binary labels validated.


In [4]:

# ML-03 — Inspect available columns for leakage risks

print("=== DATASET COLUMNS ===")
for col in df.columns:
    print(f"{col}: {df[col].dtype}")

print("\n=== POTENTIAL LEAKAGE CANDIDATES ===")
keywords = [
    "label", "target", "future", "second_half",
    "post", "outcome", "declin"
]

candidates = [
    col for col in df.columns
    if any(word in col.lower() for word in keywords)
]

print(candidates if candidates else "No keyword matches found")

print("\n=== DATASET SHAPE ===")
print(df.shape)


=== DATASET COLUMNS ===
content_id: object
client_id: object
search_volume: float64
competition: float64
competition_level: object
cpc: float64
content_type: object
main_intent: object
word_count: float64
char_count: float64
provider_used: object
model_used: object
impressions_90d: int64
clicks_90d: int64
pageviews_90d: int64
sessions_90d: int64
users_90d: int64
engaged_sessions_90d: int64
ai_sessions_90d: int64
scroll_events_90d: int64
days_with_impressions: int64
days_with_sessions: int64
impressions_last_30d: int64
clicks_last_30d: int64
sessions_last_30d: int64
impressions_prev_30d: int64
clicks_prev_30d: int64
sessions_prev_30d: int64
content_age_days: int64
age_tier: object
age_tier_order: int64
days_since_last_update: int64
freshness_tier: object
word_count_tier: object
char_count_tier: object
ctr: float64
avg_position: float64
engagement_rate: float64
scroll_rate: float64
ai_traffic_pct: float64
impression_tier: object
position_tier: object
trend_direction: object
trend_pct: fl

In [5]:

# ML-03 — Check target and trend relationship

import pandas as pd

print("=== TARGET VS TREND DIRECTION ===")

display(pd.crosstab(
    df["trend_direction"],
    df["is_declining_label"],
    margins=True
))

print("\n=== TREND PERCENTAGE BY LABEL ===")

display(
    df.groupby("is_declining_label")["trend_pct"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\n=== POSSIBLE TARGET RECONSTRUCTION ===")

# Test whether a 20% decline in trend_pct
# exactly reproduces the label.
candidate = (df["trend_pct"] < -20).astype(int)

print(
    "Agreement with 20% decline rule:",
    round((candidate == df["is_declining_label"]).mean(), 4)
)

print("\n=== SAMPLE RECORDS ===")

display(df[
    ["impressions_last_30d",
     "impressions_prev_30d",
     "trend_direction",
     "trend_pct",
     "is_declining_label"]
].head(10))


=== TARGET VS TREND DIRECTION ===


is_declining_label,0,1,All
trend_direction,,,
down,0,16262,16262
flat,1152,0,1152
new,2236,0,2236
stable,5962,0,5962
up,4388,0,4388
All,13738,16262,30000



=== TREND PERCENTAGE BY LABEL ===


,count,min,median,mean,max
is_declining_label,,,,,
0,10350,-20.0,11.3,79.003179,44900.0
1,16262,-100.0,-55.6,-58.113830,-20.0



=== POSSIBLE TARGET RECONSTRUCTION ===
Agreement with 20% decline rule: 0.9999

=== SAMPLE RECORDS ===


,impressions_last_30d,impressions_prev_30d,trend_direction,trend_pct,is_declining_label
0,578,987,down,-41.4,1
1,2501,5915,down,-57.7,1
2,2382,6089,down,-60.9,1
3,3626,4206,stable,-13.8,0
4,4211,6452,down,-34.7,1
5,617,1009,down,-38.9,1
6,1,13,down,-92.3,1
7,636,632,stable,0.6,0
8,5696,13828,down,-58.8,1
9,252,356,down,-29.2,1


In [6]:

# ML-03 — Reconstruct target from impression windows

import numpy as np

previous = df["impressions_prev_30d"]
recent = df["impressions_last_30d"]
target = df["is_declining_label"]

# Test decline definition directly from counts
reconstructed = (
    (previous > 0) &
    (recent < 0.8 * previous)
).astype(int)

print("=== TARGET RECONSTRUCTION ===")
print(
    "Agreement:",
    round((reconstructed == target).mean(), 6)
)

print(
    "Mismatched records:",
    int((reconstructed != target).sum())
)

print("\n=== MISMATCH EXAMPLES ===")
display(
    df.loc[
        reconstructed != target,
        [
            "impressions_prev_30d",
            "impressions_last_30d",
            "trend_pct",
            "trend_direction",
            "is_declining_label"
        ]
    ].head(10)
)

print("\n=== ZERO PREVIOUS IMPRESSIONS ===")
print("Rows:", int((previous == 0).sum()))

print("\n=== LABEL FROM TREND DIRECTION ===")
direction_label = (
    df["trend_direction"] == "down"
).astype(int)

print(
    "Agreement:",
    round((direction_label == target).mean(), 6)
)


=== TARGET RECONSTRUCTION ===
Agreement: 1.0
Mismatched records: 0

=== MISMATCH EXAMPLES ===


,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,is_declining_label



=== ZERO PREVIOUS IMPRESSIONS ===
Rows: 3388

=== LABEL FROM TREND DIRECTION ===
Agreement: 1.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Target Reconstruction and Leakage Audit

A target reconstruction audit was conducted on the FlyRank starter dataset containing 30,000 unique content records from 32 clients.

The binary decline label was reconstructed using the previous and recent 30-day impression counts, with a decline defined as a reduction greater than 20% relative to the previous observation window, provided previous impressions were positive.

The reconstructed labels matched all 30,000 original labels, producing 100% agreement with zero mismatches. The categorical `trend_direction` feature also reproduced the target with 100% agreement.

These findings demonstrate deterministic target leakage if the impression-window variables or derived trend indicators are used as predictors. Consequently, these variables cannot be treated as independent predictors of future decline.

The starter dataset is therefore appropriate for demonstrating target construction, leakage detection, and retrospective decline identification. A valid prospective prediction experiment requires a separately defined future outcome window and features restricted to information available before that window begins.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


A hand-written rule can only combine a few signals the way a person thought to write them down. From my Week 1 run: the baseline rule scored Precision@50 = 0.240 (~12 of the top 50 actually declining), while a random forest scored 0.740 (~37 of 50) — roughly 3x better, using the same underlying signals combined in ways a fixed rule can't express.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.